# audio_02_preprocess_16k — Preprocesado de audio a 16 kHz

Este notebook estandariza todos los audios del dataset para garantizar compatibilidad con los modelos de Deep Learning y AST.

Se realizan los siguientes pasos:
- Conversión a mono
- Re-muestreo a 16 kHz
- Recorte o padding a duración fija
- Guardado en carpeta `processed`

El resultado será un conjunto de audios homogéneo listo para entrenamiento.

## 1) Librerías necesarias

Importamos librerías para:
- Manipulación de archivos (`os`, `pathlib`)
- Procesamiento de audio (`torchaudio`, `librosa`)
- Manejo de tensores (`torch`)
- Progreso del procesamiento (`tqdm`)

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import subprocess
import shutil


## 2) Rutas del proyecto y ficheros de entrada/salida

Detectamos automáticamente la raíz del proyecto `multimodal-emocion` para que el notebook funcione aunque se ejecute desde distintas carpetas.  
Definimos las rutas:

- `RAW`: audios originales (`data/audio/raw`)
- `PROCESSED`: audios procesados (`data/audio/processed`)
- `manifest.csv`: listado original (`path`, `label`, `dataset`, etc.)
- `manifest_16k.csv`: salida con rutas procesadas (`path_16k`)

In [2]:
# Raíz del proyecto: .../multimodal-emocion
HERE = Path.cwd()
PROJECT_ROOT = HERE
while PROJECT_ROOT.name != "multimodal-emocion" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("CWD:", HERE)
print("PROJECT_ROOT:", PROJECT_ROOT)
assert PROJECT_ROOT.name == "multimodal-emocion", "No estoy dentro del proyecto multimodal-emocion (ruta incorrecta)."

DATA = PROJECT_ROOT / "data" / "audio"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"

MANIFEST_IN = DATA / "manifest.csv"
MANIFEST_16K = DATA / "manifest_16k.csv"

print("RAW:", RAW)
print("PROCESSED:", PROCESSED)
print("MANIFEST_IN:", MANIFEST_IN)
print("Existe manifest.csv:", MANIFEST_IN.exists())


CWD: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\notebooks
PROJECT_ROOT: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion
RAW: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw
PROCESSED: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\processed
MANIFEST_IN: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\manifest.csv
Existe manifest.csv: True


## 3) Cargar manifest original

Leemos `manifest.csv` para obtener todas las rutas de audio y sus etiquetas.  
Mostramos un resumen (primeras filas y conteo por dataset) para comprobar que el manifest está bien construido antes de transformar audios.

In [3]:
manifest = pd.read_csv(MANIFEST_IN)
print("Filas manifest:", len(manifest))
print(manifest.head())
print("\nDatasets:", manifest["dataset"].value_counts())


Filas manifest: 22598
                                                path    label  dataset  \
0  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  neutral  ravdess   
1  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  neutral  ravdess   
2  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  neutral  ravdess   
3  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  neutral  ravdess   
4  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  neutral  ravdess   

    speaker split  
0  actor_01   all  
1  actor_01   all  
2  actor_01   all  
3  actor_01   all  
4  actor_01   all  

Datasets: dataset
meld       13716
crema_d     7442
ravdess     1440
Name: count, dtype: int64


## 4) Verificar FFmpeg (requisito para convertir audio)

Comprobamos que `ffmpeg` está disponible en el sistema, ya que se utiliza para:
- re-muestrear a 16 kHz
- convertir a WAV
- reparar/normalizar archivos problemáticos

Si no se encuentra en el PATH, intentamos localizarlo con `where ffmpeg` (Windows).

In [4]:
# 1) buscar ffmpeg en PATH
ffmpeg_path = shutil.which("ffmpeg")

# 2) si no está, usar "where ffmpeg"
if ffmpeg_path is None:
    try:
        out = subprocess.check_output("where ffmpeg", shell=True, text=True).strip().splitlines()
        ffmpeg_path = out[0].strip() if out else None
    except Exception:
        ffmpeg_path = None

assert ffmpeg_path is not None, "No encuentro ffmpeg. Instálalo o ponlo en PATH (winget install Gyan.FFmpeg)."

FFMPEG = ffmpeg_path
print("FFMPEG detectado:", FFMPEG)
subprocess.run([FFMPEG, "-version"], check=True)
print("OK: ffmpeg funciona")


FFMPEG detectado: C:\Users\Rafa\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin\ffmpeg.EXE
OK: ffmpeg funciona


## 5) Funciones auxiliares de validación (tamaño y lectura)

Definimos comprobaciones rápidas para evitar procesar archivos corruptos o “placeholders”:

- `is_probably_audio_file`: existe y tiene tamaño razonable
- `ffprobe_is_audio_ok`: verifica que FFmpeg puede leerlo sin errores

Estas validaciones reducen fallos durante el procesado masivo.

In [5]:
def is_probably_audio_file(p: Path) -> bool:
    """Chequeo rápido: existe y pesa más que un placeholder."""
    return p.exists() and p.is_file() and p.stat().st_size > 4096  # >4KB

def ffprobe_is_audio_ok(p: Path) -> bool:
    """Comprueba si ffprobe/ffmpeg puede leer el archivo."""
    try:
        r = subprocess.run(
            [FFMPEG, "-v", "error", "-i", str(p), "-f", "null", "-"],
            capture_output=True,
            text=True
        )
        return r.returncode == 0
    except Exception:
        return False

def to_wav16k(in_path: Path, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        FFMPEG, "-hide_banner", "-y",
        "-i", str(in_path),
        "-ac", "1",
        "-ar", "16000",
        "-vn",
        str(out_path)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout, r.stderr


## 6) Estructura de salida (carpetas procesadas)

Definimos cómo se guardarán los audios procesados:

`processed/<dataset>/<split>/<stem>.wav`

Esto mantiene el dataset ordenado por origen (`dataset`) y partición (`split`), y evita nombres duplicados.

In [6]:
def out_path_for_row(row) -> Path:
    # Guardamos todo como: processed/<dataset>/<split>/<stem>.wav
    split = row["split"] if isinstance(row.get("split", None), str) else "all"
    stem = Path(row["path"]).stem
    return PROCESSED / row["dataset"] / split / f"{stem}.wav"


## 7) Procesado principal (todos los datasets excepto CREMA-D)

Procesamos el manifest fila a fila:
- convertimos cada audio a WAV
- re-muestreamos a **16 kHz**
- guardamos en `processed/...`

**Nota:** en este bucle se omite temporalmente `crema_d` para tratarlo aparte (da problemas con WAV/MP3 según la instalación del dataset).

In [7]:
new_paths = []
fails = []

total = len(manifest)
for i, row in manifest.iterrows():
    p = Path(row["path"])
    outp = out_path_for_row(row)

    # De momento saltamos crema_d, lo haremos aparte para evitar sus problemas
    if row["dataset"] == "crema_d":
        new_paths.append(None)
        continue

    try:
        if outp.exists() and outp.stat().st_size > 4096:
            new_paths.append(str(outp))
        else:
            if not p.exists():
                new_paths.append(None)
                fails.append((i, str(p), "NO_EXISTE"))
            else:
                code, so, se = to_wav16k(p, outp)
                if code == 0 and outp.exists() and outp.stat().st_size > 4096:
                    new_paths.append(str(outp))
                else:
                    new_paths.append(None)
                    fails.append((i, str(p), se.strip()[:2000]))
    except Exception as e:
        new_paths.append(None)
        fails.append((i, str(p), repr(e)))

    if (i+1) % 500 == 0:
        print(f"Progreso: {i+1}/{total} | fallos acumulados: {len(fails)}")

manifest2 = manifest.copy()
manifest2["path_16k"] = new_paths

manifest_ok = manifest2.dropna(subset=["path_16k"]).copy()
manifest_ok.to_csv(MANIFEST_16K, index=False)

print("\nConversión NO-crema terminada.")
print("Total:", total)
print("OK:", len(manifest_ok))
print("Fallos:", len(fails))
print("Guardado:", MANIFEST_16K)

if fails:
    print("\nEjemplos de fallos (hasta 10):")
    for x in fails[:10]:
        print(x)


Progreso: 500/22598 | fallos acumulados: 0
Progreso: 1000/22598 | fallos acumulados: 0
Progreso: 9000/22598 | fallos acumulados: 1
Progreso: 9500/22598 | fallos acumulados: 1
Progreso: 10000/22598 | fallos acumulados: 2
Progreso: 10500/22598 | fallos acumulados: 2
Progreso: 11000/22598 | fallos acumulados: 2
Progreso: 11500/22598 | fallos acumulados: 2
Progreso: 12000/22598 | fallos acumulados: 2
Progreso: 12500/22598 | fallos acumulados: 2
Progreso: 13000/22598 | fallos acumulados: 2
Progreso: 13500/22598 | fallos acumulados: 2
Progreso: 14000/22598 | fallos acumulados: 3
Progreso: 14500/22598 | fallos acumulados: 3
Progreso: 15000/22598 | fallos acumulados: 3
Progreso: 15500/22598 | fallos acumulados: 5
Progreso: 16000/22598 | fallos acumulados: 6
Progreso: 16500/22598 | fallos acumulados: 8
Progreso: 17000/22598 | fallos acumulados: 8
Progreso: 17500/22598 | fallos acumulados: 9
Progreso: 18000/22598 | fallos acumulados: 9
Progreso: 18500/22598 | fallos acumulados: 10
Progreso: 1900

## 8) CREMA-D — detectar fuente válida (AudioWAV vs AudioMP3)

CREMA-D puede venir con WAV “vacíos” o con MP3 como fuente fiable según la descarga.  
Aquí comprobamos qué carpeta existe y listamos los ficheros disponibles:

- `AudioWAV/*.wav`
- `AudioMP3/*.mp3`

Esto prepara la decisión de qué formato usar para el procesado.

In [8]:
crema_dir = RAW / "crema_d"
wav_dir = crema_dir / "AudioWAV"
mp3_dir = crema_dir / "AudioMP3"

print("crema_dir:", crema_dir, "exists:", crema_dir.exists())
print("wav_dir:", wav_dir, "exists:", wav_dir.exists())
print("mp3_dir:", mp3_dir, "exists:", mp3_dir.exists())

wav_files = sorted(wav_dir.glob("*.wav")) if wav_dir.exists() else []
mp3_files = sorted(mp3_dir.glob("*.mp3")) if mp3_dir.exists() else []

print("WAV encontrados:", len(wav_files))
print("MP3 encontrados:", len(mp3_files))

# Muestra tamaños (para detectar placeholders)
def show_sizes(files, n=5):
    for f in files[:n]:
        print(f.name, "size:", f.stat().st_size)

print("\nEjemplo tamaños WAV:")
show_sizes(wav_files, 5)

print("\nEjemplo tamaños MP3:")
show_sizes(mp3_files, 5)


crema_dir: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\crema_d exists: True
wav_dir: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\crema_d\AudioWAV exists: True
mp3_dir: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\crema_d\AudioMP3 exists: False
WAV encontrados: 7442
MP3 encontrados: 0

Ejemplo tamaños WAV:
1001_DFA_ANG_XX.wav size: 72862
1001_DFA_DIS_XX.wav size: 74786
1001_DFA_FEA_XX.wav size: 69446
1001_DFA_HAP_XX.wav size: 59836
1001_DFA_NEU_XX.wav size: 65176

Ejemplo tamaños MP3:


## 9) CREMA-D — elegir WAV si son “reales” (si no, usar MP3)

Si hay WAVs, comprobamos que sean “reales” por tamaño (>4KB) en una muestra.  
- Si pasan el umbral → usamos `AudioWAV`
- Si no → usamos `AudioMP3`

Con esto evitamos fallos típicos por archivos WAV corruptos o placeholders.

In [9]:
use_wav = False
if wav_files:
    # Si al menos 50 wav parecen "reales" por tamaño >4KB, los usamos
    realish = sum(1 for f in wav_files[:200] if is_probably_audio_file(f))
    use_wav = realish >= 50

if use_wav:
    CREMA_SRC_DIR = wav_dir
    CREMA_GLOB = "*.wav"
    print("✅ Usaremos CREMA desde AudioWAV (parecen reales).")
else:
    CREMA_SRC_DIR = mp3_dir
    CREMA_GLOB = "*.mp3"
    print("⚠️ AudioWAV no parece válido. Usaremos AudioMP3 si existe.")
    assert mp3_dir.exists(), "No hay AudioMP3 disponible y AudioWAV no parece válido."

print("CREMA_SRC_DIR:", CREMA_SRC_DIR)


✅ Usaremos CREMA desde AudioWAV (parecen reales).
CREMA_SRC_DIR: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\crema_d\AudioWAV


## 10) CREMA-D — procesado específico y actualización de rutas

Filtramos el manifest a `crema_d` y procesamos sus audios usando la fuente elegida (WAV o MP3).  
Registramos cuántos audios se convierten correctamente y guardamos ejemplos de fallos para depuración.

In [10]:
# cargamos manifest original para quedarnos con crema_d
manifest = pd.read_csv(MANIFEST_IN)
crema = manifest[manifest["dataset"] == "crema_d"].copy()
print("CREMA filas:", len(crema))

fails_crema = 0
ok_crema = 0
fail_examples = []

# procesamos crema usando su split (en manifest es "all")
new_paths_crema = []

for j, row in crema.iterrows():
    stem = Path(row["path"]).stem

    # archivo de entrada según fuente elegida
    in_path = CREMA_SRC_DIR / f"{stem}{Path(CREMA_GLOB.replace('*','')).suffix}"
    # pero como suffix arriba no sirve, hacemos:
    if CREMA_GLOB == "*.wav":
        in_path = CREMA_SRC_DIR / f"{stem}.wav"
    else:
        in_path = CREMA_SRC_DIR / f"{stem}.mp3"

    outp = PROCESSED / "crema_d" / "all" / f"{stem}.wav"

    # validación + conversión
    if not in_path.exists():
        new_paths_crema.append(None)
        fails_crema += 1
        if len(fail_examples) < 10:
            fail_examples.append((stem, "NO_EXISTE", str(in_path)))
        continue

    # filtro anti-placeholder
    if not is_probably_audio_file(in_path):
        new_paths_crema.append(None)
        fails_crema += 1
        if len(fail_examples) < 10:
            fail_examples.append((stem, "PLACEHOLDER/PEQUEÑO", f"size={in_path.stat().st_size}"))
        continue

    code, so, se = to_wav16k(in_path, outp)
    if code == 0 and outp.exists() and outp.stat().st_size > 4096:
        new_paths_crema.append(str(outp))
        ok_crema += 1
    else:
        new_paths_crema.append(None)
        fails_crema += 1
        if len(fail_examples) < 10:
            fail_examples.append((stem, "FFMPEG_FAIL", se.strip()[:500]))

    if ok_crema % 200 == 0 and ok_crema > 0:
        print(f"CREMA ok: {ok_crema}/{len(crema)} | fails: {fails_crema}")

crema["path_16k"] = new_paths_crema
crema_ok = crema.dropna(subset=["path_16k"]).copy()

# cargamos el manifest_16k actual (sin crema) y lo recombinamos
mf16 = pd.read_csv(MANIFEST_16K)
mf16 = mf16[mf16["dataset"] != "crema_d"].copy()

mf16_final = pd.concat([mf16, crema_ok], ignore_index=True)
mf16_final.to_csv(MANIFEST_16K, index=False)

print("\n✅ FIN CREMA con fuente:", "AudioWAV" if use_wav else "AudioMP3")
print("CREMA ok:", ok_crema)
print("CREMA fails:", fails_crema)
print("Total final manifest_16k:", len(mf16_final))
print("Guardado:", MANIFEST_16K)

if fail_examples:
    print("\nEjemplos fallos crema (hasta 10):")
    for x in fail_examples:
        print(x)


CREMA filas: 7442
CREMA ok: 200/7442 | fails: 0
CREMA ok: 400/7442 | fails: 0
CREMA ok: 600/7442 | fails: 0
CREMA ok: 800/7442 | fails: 0
CREMA ok: 1000/7442 | fails: 0
CREMA ok: 1200/7442 | fails: 0
CREMA ok: 1400/7442 | fails: 0
CREMA ok: 1600/7442 | fails: 0
CREMA ok: 1800/7442 | fails: 0
CREMA ok: 2000/7442 | fails: 0
CREMA ok: 2200/7442 | fails: 0
CREMA ok: 2400/7442 | fails: 0
CREMA ok: 2600/7442 | fails: 0
CREMA ok: 2800/7442 | fails: 0
CREMA ok: 3000/7442 | fails: 0
CREMA ok: 3200/7442 | fails: 0
CREMA ok: 3400/7442 | fails: 0
CREMA ok: 3600/7442 | fails: 0
CREMA ok: 3800/7442 | fails: 0
CREMA ok: 4000/7442 | fails: 0
CREMA ok: 4200/7442 | fails: 0
CREMA ok: 4400/7442 | fails: 0
CREMA ok: 4600/7442 | fails: 0
CREMA ok: 4800/7442 | fails: 0
CREMA ok: 5000/7442 | fails: 0
CREMA ok: 5200/7442 | fails: 0
CREMA ok: 5400/7442 | fails: 0
CREMA ok: 5600/7442 | fails: 0
CREMA ok: 5800/7442 | fails: 0
CREMA ok: 6000/7442 | fails: 0
CREMA ok: 6200/7442 | fails: 0
CREMA ok: 6400/7442 | fai

## 11) Verificación final del manifest_16k

Cargamos `manifest_16k.csv` y hacemos una comprobación rápida:
- conteo por dataset
- muestreo aleatorio de rutas (`path_16k`)
- detección de archivos que falten en disco

Este paso confirma que el preprocesado ha generado rutas válidas antes de entrenar.

In [11]:
df = pd.read_csv(MANIFEST_16K)
print("Filas manifest_16k:", len(df))
print(df["dataset"].value_counts())

# sample aleatorio
sample = df.sample(min(30, len(df)), random_state=42)
missing = []
for p in sample["path_16k"]:
    if not Path(p).exists():
        missing.append(p)

print("Missing en sample:", len(missing))
if missing:
    print(missing[:5])


Filas manifest_16k: 22586
dataset
meld       13704
crema_d     7442
ravdess     1440
Name: count, dtype: int64
Missing en sample: 0
